In [0]:
spark

In [0]:
# Replace 'your-container' with the name you found in the Azure Portal
container = "oliststorage" 
path = f"abfss://{container}@olistdataiantristan.dfs.core.windows.net/"

try:
    files = dbutils.fs.ls(path)
    display(files)
except Exception as e:
    print(f"Error: {e}")

path,name,size,modificationTime
abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/bronze/,bronze/,0,1767151869000
abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/gold/,gold/,0,1767151885000
abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/silver/,silver/,0,1767151874000


In [0]:
df = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load("abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/bronze/olist_order_payments_dataset.csv")

In [0]:
df.show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|   99.3300018|
|a9810da82917af2d9...|                 1| credit_card|                   1|   24.3899994|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|   65.7099991|
|ba78997921bbcdc13...|                 1| credit_card|                   8|   107.779999|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|   128.449997|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows


In [0]:
display(df)

In [0]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [0]:
#Reading all the files in the bronze folder

customer_df = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load("abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/bronze/olist_customers_dataset.csv")

geolocation_df = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load("abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/bronze/olist_geolocation_dataset.csv")

order_items_df = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load("abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/bronze/olist_order_items_dataset.csv")

order_payments_df = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load("abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/bronze/olist_order_payments_dataset.csv")

order_reviews_df = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load("abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/olist_order_reviews_dataset.csv")

order_df = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load("abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/bronze/olist_orders_dataset.csv")

products_df = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load("abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/bronze/olist_products_dataset.csv")

sellers_df = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load("abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/bronze/olist_sellers_dataset.csv")



In [0]:
sellers_df.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [0]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.7 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 1.7/1.7 MB 58.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/331.1 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.4 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#Mongo DB for data enrichment
from pymongo import MongoClient
import json


In [0]:
hostname = "t4sfj8.h.filess.io"
database = "Mongo_NOSQL_peacegreen"
port = "61003"
username = "Mongo_NOSQL_peacegreen"
password = "6f8c10c062905cf3bbe4e06fa035a206b1b1eb07"

url = f"mongodb://{username}:{password}@{hostname}:{port}/{database}?authSource=admin"

client = MongoClient(url)
db = client[database]
     

In [0]:
db

Database(MongoClient(host=['t4sfj8.h.filess.io:61003'], document_class=dict, tz_aware=False, connect=True, authsource='admin'), 'Mongo_NOSQL_peacegreen')

In [0]:
collection_name = 'product_category_translations'
collection = db[collection_name]

In [0]:
collection

Collection(Database(MongoClient(host=['t4sfj8.h.filess.io:61003'], document_class=dict, tz_aware=False, connect=True, authsource='admin'), 'Mongo_NOSQL_peacegreen'), 'product_category_translations')

In [0]:
import pandas as pd

mongo_df = pd.DataFrame(list(collection.find()))

In [0]:
import pymongo
from urllib.parse import quote_plus

user = "Mongo_NOSQL_peacegreen"
password = "6f8c10c062905cf3bbe4e06fa035a206b1b1eb07" # Use raw password here
host = "t4sfj8.h.filess.io:61003"
db_name = "Mongo_NOSQL_peacegreen"

# This automatically handles the special characters for you
uri = f"mongodb://{quote_plus(user)}:{quote_plus(password)}@{host}/{db_name}?authSource={db_name}"

try:
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    print(client.server_info()) # This triggers the actual connection
    print("Success!")
except Exception as e:
    print(f"Connection failed: {e}")

{'version': '5.0.32', 'gitVersion': 'ba92303e18e7ed4701572aa15acd161c97796f2f', 'modules': [], 'allocator': 'tcmalloc', 'javascriptEngine': 'mozjs', 'sysInfo': 'deprecated', 'versionArray': [5, 0, 32, 0], 'openssl': {'running': 'OpenSSL 1.1.1f  31 Mar 2020', 'compiled': 'OpenSSL 1.1.1f  31 Mar 2020'}, 'buildEnvironment': {'distmod': 'ubuntu2004', 'distarch': 'x86_64', 'cc': '/opt/mongodbtoolchain/v3/bin/gcc: gcc (GCC) 8.5.0', 'ccflags': '-Werror -include mongo/platform/basic.h -ffp-contract=off -fasynchronous-unwind-tables -ggdb -Wall -Wsign-compare -Wno-unknown-pragmas -Winvalid-pch -fno-omit-frame-pointer -fno-strict-aliasing -O2 -march=sandybridge -mtune=generic -mprefer-vector-width=128 -Wno-unused-local-typedefs -Wno-unused-function -Wno-deprecated-declarations -Wno-unused-const-variable -Wno-unused-but-set-variable -Wno-missing-braces -fstack-protector-strong -Wa,--nocompress-debug-sections -fno-builtin-memcmp', 'cxx': '/opt/mongodbtoolchain/v3/bin/g++: g++ (GCC) 8.5.0', 'cxxflag

In [0]:
database = "Mongo_NOSQL_peacegreen"

df = (spark.read
      .format("mongodb")
      .option("spark.mongodb.read.connection.uri", uri) # 
      .option("database", database)
      .option("collection", collection)
      .load())

display(df)

In [0]:
try:
    # We aren't loading data yet, just checking if the "provider" exists
    spark.read.format("mongodb").option("uri", "mongodb://test").load
    print("✅ Connector is properly registered and ready!")
except Exception as e:
    if "DATA_SOURCE_NOT_FOUND" in str(e):
        print("❌ Connector still not found. Try restarting the cluster.")
    else:
        # If we get an auth error here, it's actually GOOD news 
        # because it means the connector tried to work!
        print("✅ Connector found (received expected connection error).")

✅ Connector is properly registered and ready!


In [0]:
import pandas as pd
from pymongo import MongoClient

# 1. Get the data using Python (which works on Serverless)
client = MongoClient(uri)
db = client["Mongo_NOSQL_peacegreen"]
collection = db["product_category_translations"]

# 2. Convert to a Pandas DataFrame
data = list(collection.find())
pdf = pd.DataFrame(data)

# 3. Drop the MongoDB '_id' (Spark doesn't like ObjectId types)
if '_id' in pdf.columns:
    pdf = pdf.drop(columns=['_id'])

# 4. Convert to Spark DataFrame
df_mongo = spark.createDataFrame(pdf)

display(df_mongo)

product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor
esporte_lazer,sports_leisure
perfumaria,perfumery
utilidades_domesticas,housewares
telefonia,telephony
relogios_presentes,watches_gifts


##Cleaning Data

In [0]:
from pyspark.sql.functions import col, to_date,date_diff,current_date, when

In [0]:
def clean_data(df, df_name):
    print(f"Cleaning {df_name}")
    return df.dropDuplicates().na.drop("all")

order_df = clean_data(order_df, "order_df")



Cleaning order_df


In [0]:
order_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [0]:
#convert to Date
order_df = order_df.withColumn("order_purchase_timestamp", to_date(col("order_purchase_timestamp")))
order_df = order_df.withColumn("order_approved_at", to_date(col("order_approved_at")))
order_df = order_df.withColumn("order_delivered_carrier_date", to_date(col("order_delivered_carrier_date")))
order_df = order_df.withColumn("order_delivered_customer_date", to_date(col("order_delivered_customer_date")))
order_df = order_df.withColumn("order_estimated_delivery_date", to_date(col("order_estimated_delivery_date")))

In [0]:
order_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: date (nullable = true)
 |-- order_approved_at: date (nullable = true)
 |-- order_delivered_carrier_date: date (nullable = true)
 |-- order_delivered_customer_date: date (nullable = true)
 |-- order_estimated_delivery_date: date (nullable = true)



In [0]:
#Calculate Delivery Time and TIme Delays
order_df = order_df.withColumn("actual_delivery_time", date_diff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
order_df = order_df.withColumn("estimated_delivery_time", date_diff(col("order_estimated_delivery_date"), col("order_purchase_timestamp")))
order_df = order_df.withColumn("delay", col("actual_delivery_time") > col("estimated_delivery_time"))
order_df = order_df.withColumn("delay_time", when(col("delay") == True, col("actual_delivery_time") - col("estimated_delivery_time")).otherwise(0))
order_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: date (nullable = true)
 |-- order_approved_at: date (nullable = true)
 |-- order_delivered_carrier_date: date (nullable = true)
 |-- order_delivered_customer_date: date (nullable = true)
 |-- order_estimated_delivery_date: date (nullable = true)
 |-- actual_delivery_time: integer (nullable = true)
 |-- estimated_delivery_time: integer (nullable = true)
 |-- delay: boolean (nullable = true)
 |-- delay_time: integer (nullable = true)



In [0]:
display(order_df)

In [0]:
display(order_df.tail(5))

##Joining Data

In [0]:
order_customer_df = order_df.join(customer_df, "customer_id", "left")
order_customer_items_df = order_customer_df.join(order_items_df, "order_id", "left")
order_customer_items_payments_df = order_customer_items_df.join(order_payments_df, "order_id", "left")
order_customer_items_payments_products_df = order_customer_items_payments_df.join(products_df, "product_id", "left")
order_customer_items_payments_products_sellers_df = order_customer_items_payments_products_df.join(sellers_df, "seller_id", "left")
order_customer_items_payments_products_sellers_geolocation_df = (
    order_customer_items_payments_products_sellers_df.join(
        geolocation_df, 
        col("geolocation_zip_code_prefix") == col("seller_zip_code_prefix"), 
        "left"))

In [0]:
full_df = order_customer_items_payments_products_sellers_geolocation_df.join(order_reviews_df, "order_id", "left")


In [0]:
full_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: date (nullable = true)
 |-- order_approved_at: date (nullable = true)
 |-- order_delivered_carrier_date: date (nullable = true)
 |-- order_delivered_customer_date: date (nullable = true)
 |-- order_estimated_delivery_date: date (nullable = true)
 |-- actual_delivery_time: integer (nullable = true)
 |-- estimated_delivery_time: integer (nullable = true)
 |-- delay: boolean (nullable = true)
 |-- delay_time: integer (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (

In [0]:
#Enriching Data with MongoDB

full_df = full_df.join(df_mongo, "product_category_name", "left")
display(full_df)

product_category_name,order_id,seller_id,product_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_time,estimated_delivery_time,delay,delay_time,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,product_category_name_english
moveis_decoracao,26dbb32053df6d12465fe5eb115e484a,f00f5b35d0abcacbdd863672f4bb2c1a,71ca28a845714b71b378d81e93f0b183,88bf6f648e00e223da6d2d0cbad14ede,delivered,2017-05-27,2017-05-27,2017-06-02,2017-06-08,2017-06-29,12,33,false,0,95a83038c5ef655802b77358c6c0f204,18017,sorocaba,SP,1,2017-06-13T14:25:16Z,299.0,14.43,1,credit_card,6,313.429993,27,373,2,800,63,6,11,1238,sao paulo,SP,1238,-23.54195269441404,-46.65931762155928,sao paulo,SP,59dfec4d0c363b2f4bce64dfeb778764,4,null,null,2017-06-09 00:00:00,2017-06-10 11:31:16,furniture_decor
moveis_decoracao,26dbb32053df6d12465fe5eb115e484a,f00f5b35d0abcacbdd863672f4bb2c1a,71ca28a845714b71b378d81e93f0b183,88bf6f648e00e223da6d2d0cbad14ede,delivered,2017-05-27,2017-05-27,2017-06-02,2017-06-08,2017-06-29,12,33,false,0,95a83038c5ef655802b77358c6c0f204,18017,sorocaba,SP,1,2017-06-13T14:25:16Z,299.0,14.43,1,credit_card,6,313.429993,27,373,2,800,63,6,11,1238,sao paulo,SP,1238,-23.54443774132954,-46.65181342394259,sao paulo,SP,59dfec4d0c363b2f4bce64dfeb778764,4,null,null,2017-06-09 00:00:00,2017-06-10 11:31:16,furniture_decor
moveis_decoracao,26dbb32053df6d12465fe5eb115e484a,f00f5b35d0abcacbdd863672f4bb2c1a,71ca28a845714b71b378d81e93f0b183,88bf6f648e00e223da6d2d0cbad14ede,delivered,2017-05-27,2017-05-27,2017-06-02,2017-06-08,2017-06-29,12,33,false,0,95a83038c5ef655802b77358c6c0f204,18017,sorocaba,SP,1,2017-06-13T14:25:16Z,299.0,14.43,1,credit_card,6,313.429993,27,373,2,800,63,6,11,1238,sao paulo,SP,1238,-23.542944348058064,-46.65793705274388,sao paulo,SP,59dfec4d0c363b2f4bce64dfeb778764,4,null,null,2017-06-09 00:00:00,2017-06-10 11:31:16,furniture_decor
moveis_decoracao,26dbb32053df6d12465fe5eb115e484a,f00f5b35d0abcacbdd863672f4bb2c1a,71ca28a845714b71b378d81e93f0b183,88bf6f648e00e223da6d2d0cbad14ede,delivered,2017-05-27,2017-05-27,2017-06-02,2017-06-08,2017-06-29,12,33,false,0,95a83038c5ef655802b77358c6c0f204,18017,sorocaba,SP,1,2017-06-13T14:25:16Z,299.0,14.43,1,credit_card,6,313.429993,27,373,2,800,63,6,11,1238,sao paulo,SP,1238,-23.54339898029315,-46.65151833472787,sao paulo,SP,59dfec4d0c363b2f4bce64dfeb778764,4,null,null,2017-06-09 00:00:00,2017-06-10 11:31:16,furniture_decor
moveis_decoracao,26dbb32053df6d12465fe5eb115e484a,f00f5b35d0abcacbdd863672f4bb2c1a,71ca28a845714b71b378d81e93f0b183,88bf6f648e00e223da6d2d0cbad14ede,delivered,2017-05-27,2017-05-27,2017-06-02,2017-06-08,2017-06-29,12,33,false,0,95a83038c5ef655802b77358c6c0f204,18017,sorocaba,SP,1,2017-06-13T14:25:16Z,299.0,14.43,1,credit_card,6,313.429993,27,373,2,800,63,6,11,1238,sao paulo,SP,1238,-23.54323868858565,-46.65731599111321,sao paulo,SP,59dfec4d0c363b2f4bce64dfeb778764,4,null,null,2017-06-09 00:00:00,2017-06-10 11:31:16,furniture_decor
moveis_decoracao,26dbb32053df6d12465fe5eb115e484a,f00f5b35d0abcacbdd863672f4bb2c1a,71ca28a845714b71b378d81e93f0b183,88bf6f648e00e223da6d2d0cbad14ede,delivered,2017-05-27,2017-05-27,2017-06-02,2017-06-08,2017-06-29,12,33,false,0,95a83038c5ef655802b77358c6c0f204,18017,sorocaba,SP,1,2017-06-13T14:25:16Z,299.0,14.43,1,credit_card,6,313.429993,27,373,2,800,63,6,11,1238,sao paulo,SP,1238,-23.544121128772638,-46.65531989785911,sao 

Databricks visualization. Run in Databricks to view.

In [0]:
#Saving the transformed data to silver layer
full_df.write.mode("overwrite").parquet("abfss://oliststorage@olistdataiantristan.dfs.core.windows.net/silver/")